# Interpreting GPT-2 Hidden Layers with Sparse Autoencoders (SAE)

## Complete Tutorial: From Paper to Practice

This notebook implements the methodology from **"Towards Monosemanticity: Decomposing Language Models With Dictionary Learning"** to interpret what GPT-2 learns in its hidden layers.

### What You'll Learn:

1. **Extract activations** from GPT-2's hidden layers
2. **Train a Sparse Autoencoder** to decompose these activations into interpretable features
3. **Analyze the features** to understand what concepts GPT-2 represents internally
4. **Visualize** which features activate for different inputs

### Key Concepts:

- **Hidden Layer Activations**: The intermediate representations GPT-2 computes for each token
- **Sparse Autoencoders**: Neural networks that learn sparse, interpretable decompositions
- **Monosemanticity**: The goal of having each feature represent a single, interpretable concept
- **Feature Analysis**: Understanding what linguistic or semantic patterns activate each feature

---

### Prerequisites:
- Basic PyTorch knowledge
- Understanding of transformers (helpful but not required)
- GPU recommended (but CPU works for smaller experiments)

## Step 1: Install and Import Required Libraries

First, we need to install all necessary packages. If you're running this for the first time, uncomment the pip install line.

In [ ]:
# Uncomment to install required packages
# !pip install torch transformers datasets numpy matplotlib seaborn tqdm scikit-learn einops

In [ ]:
# Import standard libraries
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Import PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

# Import HuggingFace libraries
from transformers import GPT2LMHeadModel, GPT2Tokenizer, GPT2Config
from datasets import load_dataset

# Add our src directory to path
sys.path.append('../src')

# Import our custom modules
from data_collection import GPT2ActivationCollector, prepare_training_data
from sae import SparseAutoencoder, create_sae_for_gpt2
from training import SAETrainer, train_sae
from interpretation import FeatureAnalyzer

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Display versions
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Step 2: Load GPT-2 Model and Tokenizer

We'll load the base GPT-2 model from HuggingFace. This model has:
- **12 transformer layers** (layers 0-11)
- **768-dimensional hidden states** at each layer
- **50,257 vocabulary tokens**

### Why GPT-2?
- Well-documented architecture
- Moderate size (124M parameters)
- Publicly available
- Good for learning interpretability techniques

In [ ]:
# Load GPT-2 tokenizer and model
print("Loading GPT-2 model...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
# GPT-2 doesn't have a padding token by default, so we set it to eos_token
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(
    "gpt2",
    output_hidden_states=True  # CRITICAL: enables access to intermediate layers
).to(device)

# Set to evaluation mode (disables dropout, etc.)
model.eval()

# Get model configuration
config = model.config
print(f"\n✓ Model loaded successfully!")
print(f"  Model: gpt2 (GPT-2 Small)")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Hidden size (d_model): {config.hidden_size}")
print(f"  Number of layers: {config.n_layer}")
print(f"  Number of attention heads: {config.n_head}")
print(f"  Vocabulary size: {config.vocab_size}")

## Step 3: Understanding GPT-2 Architecture and Hidden Layers

Let's explore GPT-2's structure to understand what we're interpreting.

### Transformer Architecture:

```
Input Tokens → Embedding → [Transformer Block] × 12 → Output Logits
                               ↑
                          Hidden States (what we want to interpret!)
```

Each transformer block contains:
- **Multi-head self-attention**: How tokens attend to each other
- **Feed-forward network**: Non-linear transformations
- **Layer normalization**: Stabilizes training

The **hidden states** at each layer represent what the model "thinks" about each token at that point in processing.

In [ ]:
# Let's examine a sample forward pass to see the hidden states
sample_text = "The quick brown fox jumps over the lazy dog."
print(f"Sample text: {sample_text}")

# Tokenize
inputs = tokenizer(sample_text, return_tensors="pt").to(device)
print(f"\nTokenized: {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")
print(f"Token IDs: {inputs['input_ids'][0].tolist()}")
print(f"Number of tokens: {inputs['input_ids'].shape[1]}")

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# Examine hidden states
hidden_states = outputs.hidden_states
print(f"\n✓ Number of layer outputs: {len(hidden_states)}")
print(f"  (1 embedding layer + {config.n_layer} transformer layers)")

# Each hidden state has shape: (batch_size, sequence_length, hidden_size)
for i, hidden_state in enumerate(hidden_states):
    layer_name = "Embedding" if i == 0 else f"Layer {i-1}"
    print(f"  {layer_name}: shape {tuple(hidden_state.shape)}")

print(f"\n💡 We'll focus on Layer 8 (middle layer) for interpretability")
print(f"   Middle layers often contain the most semantic information!")

## Step 4: Extract Activations from Layer 8

Now we'll collect activations from Layer 8 across many text samples. This gives us training data for the Sparse Autoencoder.

### Why collect many activations?
- Need diversity to learn general features
- Each token position gives us one activation vector
- More data = better feature discovery

### What layer to choose?
- **Early layers (0-3)**: Basic syntax, word identities  
- **Middle layers (4-8)**: Semantic features, relationships ✓ Best for interpretation
- **Late layers (9-11)**: Task-specific, prediction-focused

In [ ]:
# Initialize activation collector for Layer 8
TARGET_LAYER = 8

collector = GPT2ActivationCollector(
    model_name="gpt2",
    layer_index=TARGET_LAYER,
    device=device
)

print(f"✓ Activation collector initialized")
print(f"  Target layer: {TARGET_LAYER}")
print(f"  Hidden dimension: {collector.hidden_size}")

## Step 5 : Dataset

### Step 5.1: Collect activations from a text dataset

We'll use openwebtext (similar to GPT-2's training data) to collect diverse activations.

In [ ]:
# For this tutorial, we'll use a smaller dataset for faster processing
# For production use, increase num_texts and max_samples

print("Collecting activations from openwebtext ...")
print("This may take a few minutes...\n")

# Collect activations
# Adjust these parameters based on your compute resources:
# - More texts = more diverse features
# - More samples = better training
# - Recommended: 10000+ texts, 100000+ samples for production

activations = collector.collect_from_dataset(
    dataset_name="openwebtext",
    split="train",
    num_texts=500,        # (use 5000+ for production)
    max_samples=20000,    # Total activation vectors (use 100000+ for production)
    batch_size=16,
    max_length=128
)

print(f"\n✓ Activation collection complete!")
print(f"  Shape: {activations.shape}")
print(f"  Memory: {activations.element_size() * activations.nelement() / 1024**2:.2f} MB")

### Step 5.1: Prepare Training Data for SAE

Split the data into training and validation sets, and normalize for stable training.

In [ ]:
# Prepare data for training
train_data, val_data, stats = prepare_training_data(
    activations,
    train_ratio=0.9,  # 90% train, 10% validation
    normalize=True     # Center and scale activations
)

print("\n✓ Data preparation complete!")
print(f"  Training samples: {train_data.shape[0]:,}")
print(f"  Validation samples: {val_data.shape[0]:,}")
print(f"  Feature dimension: {train_data.shape[1]}")
print(f"\n  Normalization stats:")
print(f"    Mean: ~0.0 (centered)")
print(f"    Std: ~1.0 (scaled)")

## Step 6: Implement the Sparse Autoencoder Architecture

The SAE has a simple but powerful architecture:

### Mathematical Formulation:
```
Given activation x ∈ ℝ^768:
1. Pre-center:    x' = x - b_pre
2. Encode:        f = ReLU(W_enc @ x' + b_enc)     [768 → 12288]
3. Decode:        x̂ = W_dec @ f + b_dec            [12288 → 768]
4. Loss:          L = ||x - x̂||² + λ||f||₁
```

### Key hyperparameters:
- **Expansion factor (16x)**: 768 → 12,288 features
  - Larger = more capacity for diverse features
  - Typical range: 8x to 64x
  
- **L1 coefficient (3e-4)**: Controls sparsity
  - Larger = sparser features (more zeros)
  - Smaller = denser features (better reconstruction)
  
### Why ReLU activation?
- Enforces non-negativity (f ≥ 0)
- Natural sparsity (many values become exactly 0)
- Interpretable: feature strength ∝ activation value

In [ ]:
# Create Sparse Autoencoder
EXPANSION_FACTOR = 16  # 768 * 16 = 12,288 features
L1_COEFFICIENT = 3e-4  # Balance between reconstruction and sparsity

sae = create_sae_for_gpt2(
    model_name="gpt2",
    expansion_factor=EXPANSION_FACTOR,
    l1_coeff=L1_COEFFICIENT,
    use_tied_weights=False,      # Independent encoder/decoder
    use_pre_bias=True,            # Pre-encoding centering
    normalize_decoder=True        # Prevent feature scaling issues
)

print(f"\n✓ Sparse Autoencoder created!")
print(f"  Input dimension: {sae.d_model}")
print(f"  Hidden dimension: {sae.d_hidden}")
print(f"  Expansion factor: {EXPANSION_FACTOR}x")
print(f"  L1 coefficient: {L1_COEFFICIENT}")
print(f"  Total parameters: {sum(p.numel() for p in sae.parameters()):,}")

# Test forward pass
with torch.no_grad():
    test_input = train_data[:32].to(device)
    test_output, test_loss, loss_dict = sae(test_input, return_loss_components=True)
    
print(f"\n✓ Test forward pass successful!")
print(f"  Input shape: {test_input.shape}")
print(f"  Output shape: {test_output.shape}")
print(f"  Initial feature density: {loss_dict['frac_active']:.2%}")

## Step 7: Train the Sparse Autoencoder

Now we'll train the SAE to learn interpretable features!

### Training Strategy:
1. **Reconstruction loss (MSE)**: Make sure we can reconstruct the original activations
2. **Sparsity penalty (L1)**: Force most features to be zero
3. **Decoder normalization**: Prevent features from growing unbounded
4. **Early stopping**: Stop when validation loss stops improving

### What to monitor:
- **Loss**: Should decrease steadily
- **Feature density**: Target 1-5% (most features zero)
- **Reconstruction quality**: Should stay high (>0.9 cosine similarity)

In [ ]:
# Create trainer
trainer = SAETrainer(
    model=sae,
    train_data=train_data,
    val_data=val_data,
    lr=1e-3,              # Learning rate (Adam default)
    batch_size=256,       # Batch size (adjust for your GPU memory)
    device=device,
    checkpoint_dir="../checkpoints"
)

print("✓ Trainer initialized")
print(f"  Optimizer: Adam (lr={trainer.lr})")
print(f"  Batch size: {trainer.batch_size}")
print(f"  Training batches per epoch: {len(trainer.train_loader)}")
print(f"  Validation batches: {len(trainer.val_loader)}")

In [ ]:
# Train the model
# Adjust num_epochs based on your time/compute constraints
# For production: 50-100 epochs
# For quick testing: 10-20 epochs

print("Starting training...")
print("This will take a while (10-30 minutes depending on data size and hardware)\n")

history = trainer.train(
    num_epochs=30,                    # Maximum epochs
    early_stopping_patience=5,       # Stop if no improvement for 5 epochs
    save_every=10,                   # Save checkpoint every 10 epochs
    log_every=1                      # Log metrics every epoch
)

print("\n🎉 Training complete!")

In [ ]:
# Plot training history
trainer.plot_training_history(save_path="../checkpoints/training_curves.png")

# Display final metrics
print("\nFinal Training Metrics:")
print(f"  Final train loss: {history['train_loss'][-1]:.6f}")
print(f"  Final val loss: {history['val_loss'][-1]:.6f}")
print(f"  Best val loss: {trainer.best_val_loss:.6f}")
print(f"  Final feature density: {history['feature_density'][-1]:.2%}")
print(f"  Training epochs: {len(history['train_loss'])}")

## Step 8: Analyze Learned Sparse Features

Now comes the exciting part - understanding what the SAE learned!

### What makes a good feature?
1. **Sparse**: Activates rarely (1-5% of the time)
2. **Monosemantic**: Represents one clear concept
3. **Consistent**: Activates on similar inputs

Let's create a feature analyzer to explore the learned features.

In [ ]:
# Create feature analyzer
analyzer = FeatureAnalyzer(
    sae=sae,
    tokenizer=tokenizer,
    device=device
)

# Generate comprehensive analysis report
# print("Generating analysis report...\n")
# analyzer.create_summary_report(
#     activations=val_data,
#     texts=[f"Sample {i}" for i in range(len(val_data))],  # Placeholder texts
#     save_path="../checkpoints/sae_analysis_report.txt"
# )

### Analyze dead features

Some features may never activate ("dead features"). Let's identify them.

In [ ]:
# Analyze dead features
dead_analysis = analyzer.analyze_dead_features(val_data)

print(f"Dead Features Analysis:")
print(f"=" * 60)
print(f"  Total features: {sae.d_hidden}")
print(f"  Dead features: {dead_analysis['num_dead']} ({dead_analysis['frac_dead']:.1%})")
print(f"  Active features: {sae.d_hidden - dead_analysis['num_dead']}")

if dead_analysis['frac_dead'] > 0.3:
    print(f"\n⚠️  High dead feature rate (>30%)")
    print(f"  Consider: Lower L1 coefficient or longer training")
elif dead_analysis['frac_dead'] > 0.1:
    print(f"\n⚠️  Moderate dead feature rate (>10%)")
    print(f"  This is acceptable but could be improved")
else:
    print(f"\n✓ Low dead feature rate (<10%) - Good!")

## Step 9: Interpret Features with Real Text Examples

Now let's see what our learned features actually respond to by feeding real text through the model:

In [ ]:
# Prepare test texts to understand what features activate on
test_texts = [
    "The cat sat on the mat.",
    "Quantum mechanics is a fundamental theory in physics.",
    "I love eating pizza and pasta.",
    "The stock market crashed yesterday.",
    "She plays the violin beautifully."
]

print("Collecting activations for test texts...")
test_activations = []
for text in test_texts:
    acts = collector.collect_activations([text])
    test_activations.append(acts)
    
test_activations = torch.cat(test_activations, dim=0)
print(f"Test activations shape: {test_activations.shape}")

In [ ]:
# Encode test activations with trained SAE
sae.eval()
with torch.no_grad():
    test_features = sae.encode(test_activations.to(device))
    
# Find top activating features for each text
top_k = 5
for i, text in enumerate(test_texts):
    feature_vals = test_features[i].cpu()
    top_features = torch.topk(feature_vals, top_k)
    
    print(f"\nText: '{text}'")
    print("Top activating features:")
    for feat_idx, feat_val in zip(top_features.indices, top_features.values):
        print(f"  Feature {feat_idx.item()}: {feat_val.item():.4f}")

## Step 10: Visualize Feature Activations

Create comprehensive dashboards to understand individual features:

In [ ]:
# Create dashboards for interesting features
feature_indices = [0, 100, 500, 1000, 5000]

# Create placeholder texts for the activations
placeholder_texts = [f"Sample {i}" for i in range(len(train_data))]


for feat_idx in feature_indices:    
    analyzer.create_feature_dashboard(feat_idx, train_data, placeholder_texts)
    print(f"\n=== Feature {feat_idx} Dashboard ===")

In [ ]:
# Compute feature correlations to find related features
print("Computing feature correlations...")

# Encode train_data to get features
with torch.no_grad():
    train_features = sae.encode(train_data.to(device)).cpu()

sample_size = min(1000, len(train_features))
sample_features = train_features[:sample_size]

# Calculate correlation matrix for a subset of features
feature_subset = [0, 50, 100, 150, 200, 250, 300, 350, 400, 450]
subset_features = sample_features[:, feature_subset]

import numpy as np
correlation_matrix = np.corrcoef(subset_features.T)

# Plot correlation heatmap
plt.figure(figsize=(10, 8))
plt.imshow(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Correlation')
plt.title('Feature Correlation Matrix (Subset)')
plt.xlabel('Feature Index')
plt.ylabel('Feature Index')
plt.xticks(range(len(feature_subset)), feature_subset, rotation=45)

plt.yticks(range(len(feature_subset)), feature_subset)
print(f"Max correlation (excluding diagonal): {np.max(correlation_matrix - np.eye(len(feature_subset))):.4f}")

plt.tight_layout()
plt.show()

## Step 11: Evaluate Reconstruction Quality

Compare original vs reconstructed activations to validate the SAE:

In [ ]:
# Reconstruct some sample activations
num_samples = 5
sample_activations = val_data[:num_samples].to(device)

with torch.no_grad():
    reconstructed, _, _ = sae(sample_activations)
    
# Calculate per-sample reconstruction error
for i in range(num_samples):
    original = sample_activations[i].cpu()
    reconstructed_sample = reconstructed[i].cpu()
    
    mse = torch.mean((original - reconstructed_sample) ** 2).item()
    cosine_sim = torch.nn.functional.cosine_similarity(
        original.unsqueeze(0), 
        reconstructed_sample.unsqueeze(0)
    ).item()
    
    print(f"\nSample {i}:")
    print(f"  MSE: {mse:.6f}")
    print(f"  Cosine Similarity: {cosine_sim:.6f}")

### Check reconstruction quality

How well does the SAE reconstruct the original activations?

In [ ]:
# Compute reconstruction error per dimension
errors = (sample_activations - reconstructed).cpu().abs().mean(dim=0)

plt.figure(figsize=(12, 4))
plt.plot(errors.numpy())
plt.xlabel('Dimension')
plt.ylabel('Mean Absolute Error')
plt.title('Reconstruction Error per Dimension')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean reconstruction error: {errors.mean().item():.6f}")
print(f"Max reconstruction error: {errors.max().item():.6f}")
print(f"Dimensions with error > 0.1: {(errors > 0.1).sum().item()}")

### Visualize reconstruction errors

Plot the distribution of reconstruction errors across dimensions:

In [ ]:
# Evaluate reconstruction quality
reconstruction_metrics = analyzer.get_reconstruction_quality(val_data)

print("Reconstruction Quality Metrics:")
print("=" * 60)
for metric, value in reconstruction_metrics.items():
    print(f"  {metric:.<30} {value:.6f}")

print("\n💡 Interpretation:")
print(f"  • MSE: Lower is better (how much error in reconstruction)")
print(f"  • Cosine similarity: Higher is better (direction preserved)")
print(f"    Target: >0.90 for good reconstruction")
print(f"  • Explained variance: Higher is better (captured information)")
print(f"    Target: >0.85")
print(f"  • Feature density: Lower is better (more interpretable)")
print(f"    Target: 1-5%")

## Step 12: Feature Analysis - Monosemanticity

Test whether features are monosemantic (respond to single concepts):

In [ ]:
# Test features on specific concepts
concept_tests = {
    "numbers": ["one", "two", "three", "123", "456"],
    "animals": ["cat", "dog", "bird", "elephant", "fish"],
    "colors": ["red", "blue", "green", "yellow", "purple"],
    "emotions": ["happy", "sad", "angry", "excited", "calm"]
}

print("Testing feature responses to specific concepts:\n")

for concept, words in concept_tests.items():
    concept_activations = []
    for word in words:
        acts = collector.collect_activations([word])
        concept_activations.append(acts)
    
    concept_activations = torch.cat(concept_activations, dim=0).to(device)
    
    with torch.no_grad():
        concept_features = sae.encode(concept_activations)
    
    # Find features that consistently activate for this concept
    mean_activation = concept_features.mean(dim=0)
    top_features = torch.topk(mean_activation, 3)
    
    print(f"Concept: {concept}")
    print(f"  Top consistent features: {top_features.indices.tolist()}")
    print(f"  Average activations: {[f'{v:.4f}' for v in top_features.values.tolist()]}\n")

### Look for concept-specific features

Analyze which features are most selective to specific concepts:

In [ ]:
# Measure feature selectivity
# A feature is selective if it activates strongly for one concept but not others

all_concept_features = {}
for concept, words in concept_tests.items():
    concept_activations = []
    for word in words:
        acts = collector.collect_activations([word])
        concept_activations.append(acts)
    
    concept_activations = torch.cat(concept_activations, dim=0).to(device)
    with torch.no_grad():
        features = sae.encode(concept_activations)
    all_concept_features[concept] = features.mean(dim=0)

# Find features with high selectivity (high activation for one concept, low for others)
print("Most selective features:\n")
for concept in concept_tests.keys():
    concept_feats = all_concept_features[concept]
    other_feats = torch.stack([all_concept_features[c] for c in concept_tests.keys() if c != concept]).mean(dim=0)
    
    selectivity = concept_feats - other_feats
    top_selective = torch.topk(selectivity, 3)
    
    print(f"{concept.capitalize()} features:")
    for idx, score in zip(top_selective.indices, top_selective.values):
        print(f"  Feature {idx.item()}: selectivity score {score.item():.4f}")
    print()

## Save final model

In [ ]:
# Save the final trained model
import os
save_dir = "../checkpoints"
os.makedirs(save_dir, exist_ok=True)

final_save_path = os.path.join(save_dir, 'sae_final.pt')
torch.save({
    'model_state_dict': sae.state_dict(),
    'training_history': history,
    'hyperparameters': {
        'input_dim': sae.d_model,
        'hidden_dim': sae.d_hidden,
        'l1_coefficient': L1_COEFFICIENT,
        'expansion_factor': EXPANSION_FACTOR,
        'target_layer': TARGET_LAYER
    }
}, final_save_path)

print(f"{'='*60}")

print(f"\n{'='*60}")
print(f"Tutorial complete! Final model saved to: {final_save_path}")

## Interpreting Results

### What We've Learned:

1. **Training Performance**: The SAE successfully learned to reconstruct GPT-2's hidden layer activations with minimal error while enforcing sparsity.

2. **Feature Discovery**: We extracted sparse features (default 12,288 features from 768 dimensions) that represent interpretable components of the model's internal representations.

3. **Sparsity**: The L1 penalty ensures only a small subset of features activate for any given input, making interpretation tractable.

4. **Monosemanticity**: By analyzing feature activations on specific concepts, we can identify whether features respond to single, interpretable concepts.

5. **Dead Features**: Some features may not activate frequently, indicating they might be redundant or require different training data.

### Next Steps:

- **Scaling**: Try different expansion factors (e.g., 32x instead of 16x)
- **Layer Comparison**: Compare features across different GPT-2 layers
- **Feature Intervention**: Modify feature activations and observe output changes
- **Automated Interpretation**: Use GPT-4 to automatically label features based on max-activating examples

## Summary

**Congratulations!** You've successfully:

✅ Loaded GPT-2 and extracted hidden layer activations  
✅ Built a Sparse Autoencoder with 16x expansion  
✅ Trained the SAE with L1 sparsity regularization  
✅ Analyzed learned features for interpretability  
✅ Tested feature monosemanticity on real concepts  
✅ Evaluated reconstruction quality  

You now have a trained SAE that can help interpret GPT-2's internal representations. The sparse features you've discovered provide insights into what the model has learned and how it processes information.

For more advanced usage, see:
- `../GUIDE.md` for detailed explanations
- `../src/` for the complete implementation
- `../run_sae.py` for command-line training

**Key Paper Reference**: This implementation is based on "Towards Monosemanticity: Decomposing Language Models With Dictionary Learning" by Anthropic.